# 국민연금 2027 목표비중 — 통합 재현 노트북

이 노트북은 **고정된 2016-09~2026-08 프록시 월수익률 입력**에서 시작하여 STEP 2부터 최종 제출 CSV까지 재생성한다. 핵심 실행에는 yfinance 네트워크 접속이 필요하지 않는다.

- Yahoo Finance는 고정 프록시 가격자료의 원출처로 공개한다.
- official mapped의 DM/EM 92:8 및 대체 1/3씩은 CMA와 독립적인 팀 중립 매핑 규칙이다.
- EM/PE/인프라/PD 10% 상한은 공식 NPS 한도가 아니라 팀의 세부 집중도 가정이다.
- 현금 0.1% 고정을 baseline으로 하고 0~2% 최적화는 sensitivity로 별도 계산한다.
- Table B의 Michaud는 순수 resampling이며 LW+Michaud는 sensitivity다.


In [ ]:
from pathlib import Path
import runpy, json
import pandas as pd
from IPython.display import display

HERE=Path.cwd().resolve()
if (HERE/"src").exists():
    ROOT=HERE
elif (HERE.parent/"src").exists():
    ROOT=HERE.parent
else:
    raise FileNotFoundError("Run from repository root or submission/ directory.")

RESULTS=ROOT/"results"
SUBMISSION=ROOT/"submission"
RESULTS.mkdir(exist_ok=True); SUBMISSION.mkdir(exist_ok=True)

def show_csv(rel,n=30):
    p=ROOT/rel
    print("\n",rel)
    df=pd.read_csv(p)
    display(df.head(n))
    return df

print("ROOT =",ROOT)


## STEP 2 — 고정 프록시 수익률 → 상관 → CMA 공분산

원자료는 Yahoo Finance에서 추출한 프록시 가격을 기반으로 하지만, 분석 실행은 저장소에 고정된 월수익률을 사용한다.


In [ ]:
runpy.run_path(str(ROOT/"src/step2_build_corr.py"),run_name="__main__")
show_csv("data/diagnostics.csv")


## STEP 5~8 — 기준 MVO, Ledoit–Wolf, Box, Ellipsoid


In [ ]:
runpy.run_path(str(ROOT/"src/analyze_steps5_8.py"),run_name="__main__")
show_csv("results/step5_mvo.csv")
show_csv("results/step6_lw_diagnostics.csv")
show_csv("results/step7_box.csv")
show_csv("results/step8_ellipsoid.csv")


## STEP 9 — 순수 Michaud 300회

Table B baseline은 각 draw의 **표본상관 × CMA σ**를 사용하는 순수 Michaud다. LW+Michaud는 별도 sensitivity로 저장한다.


In [ ]:
runpy.run_path(str(ROOT/"src/step9_michaud.py"),run_name="__main__")
show_csv("results/step9_michaud_summary.csv")
show_csv("results/step9_michaud_lw_sensitivity.csv")


## STEP 10 — 방법론 종합비교

모든 방법론의 비교 위험지표는 공통 CMA Σ 기준으로 평가하고, LW 자체 covariance 기준 위험은 보조열로 유지한다.


In [ ]:
runpy.run_path(str(ROOT/"src/step10_method_synthesis.py"),run_name="__main__")
show_csv("results/step10_tableB_weights.csv")
show_csv("results/step10_tableB_metrics.csv")
show_csv("results/step10_mu_50bp_sensitivity_summary.csv")
show_csv("results/step10_small_eigenvectors.csv")


## STEP 11 — 독립적인 w2027 Team

Team 목표는 특정 최적화 결과를 복사하지 않고 2026 실제비중, 실행가능성, 대체 비유동성, KRW 기준 해외위험을 추가 판단한다.


In [ ]:
runpy.run_path(str(ROOT/"src/step11_team_target.py"),run_name="__main__")
show_csv("results/step11_tableC_detailed.csv")
show_csv("results/step11_tableC_common.csv")
show_csv("results/step11_transition_2026H1_to_team.csv")


## STEP 12 — Policy Black–Litterman

Prior는 official mapped다. δ=2.5, τ=0.025, P=I. T=10은 forward-looking CMA의 실제 표본크기가 아니라 120개월 risk sample을 이용해 Ω=Σ/T를 구현하는 **대용 가정**이며 T=5/10/20 민감도를 함께 계산한다.


In [ ]:
runpy.run_path(str(ROOT/"src/step12_policy_bl.py"),run_name="__main__")
show_csv("results/step12_tableD.csv")
show_csv("results/step12_allocations.csv")
show_csv("results/step12_T_sensitivity.csv")
show_csv("results/step12_cash_sensitivity.csv")
show_csv("results/step12_TE095_sensitivity.csv")


## 보조 민감도 — Box·Ellipsoid·10% 상세상한


In [ ]:
runpy.run_path(str(ROOT/"src/audit_sensitivities.py"),run_name="__main__")
show_csv("results/audit_method_sensitivities.csv")
show_csv("results/audit_michaud_relaxed_caps.csv")


## STEP 13 — Stress 1~3

1. Stress 1: BL active 방향에 불리한 **CMA 직접 1SE 실패**, SE=√diag(Ω), fixed weights.
2. Stress 2: 정책 주식(국내/DM/EM) -2%p 및 상관 +0.15. PE 포함은 sensitivity.
3. Stress 3: 대체 σ×1.5, 대체-글로벌주식 상관 +0.20. eigen-clipping과 Higham을 모두 보고한다.


In [ ]:
runpy.run_path(str(ROOT/"src/step13_stress_test.py"),run_name="__main__")
show_csv("results/step13_stress1_direct_cma.csv")
show_csv("results/step13_stress_results.csv")
show_csv("results/step13_covariance_diagnostics.csv")
show_csv("results/step13_T_stress1_sensitivity.csv")


## STEP 14 — 후보 비교 → 최종 IC 의결 대기\n\n최종 후보는 **Team, baseline Robust BL(TE 1.0%), 50:50 Team/Robust BL committee overlay** 세 가지다. TE 0.95% Robust BL과 25/75 blend는 민감도로만 유지한다. 현재 `FINAL_CANDIDATE=None`으로 최종안은 미정이며, 공통 기준 비교와 stress 결과를 검토한 뒤 위원회 판단으로 확정한다. `step14_decision_aid.csv`의 손익분기 확률은 시나리오 확률을 추정한 값이 아니라 판단 보조자료다.\n

In [ ]:
runpy.run_path(str(ROOT/"src/step14_candidate_comparison.py"),run_name="__main__")\nshow_csv("results/step14_candidate_comparison.csv")\nshow_csv("results/step14_decision_aid.csv")\nrunpy.run_path(str(ROOT/"src/step14_final_decision.py"),run_name="__main__")\nshow_csv("results/step14_final_weights.csv")\nshow_csv("submission/team2_views.csv")\nprint((RESULTS/"step14_final_resolution.json").read_text(encoding="utf-8"))\n